# Ü-F — ResolveChoice: die vier Strategien

`loyalty_points` in `customers.json` kommt mal als Zahl (`1200`), mal als Text
(`"gold"`) vor. Ein DynamicFrame kodiert das als **`choice<long,string>`**.
`ResolveChoice` bietet vier Wege, den Konflikt aufzulösen — dieses Notebook
stellt sie **nebeneinander**, damit du die Schemata vergleichen kannst.

Die Zellen sind vorgefertigt — **ersetze nur die `____`**. Die Syntax steht schon;
du entscheidest je Zelle, WELCHE Strategie zum beschriebenen Verhalten passt.
Der Werkzeugkasten (die vier gültigen Strategie-Strings, Reihenfolge absichtlich
gemischt):

> `make_struct` · `project:<typ>` · `make_cols` · `cast:<typ>`

Ordne jeder Zelle den passenden String zu. Für `<typ>` setzt du den Zieltyp ein.

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
%%configure
{ "--enable-glue-datacatalog": "true" }

In [ ]:
from awsglue.context import GlueContext
from awsglue.transforms import ResolveChoice
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

# Bucket-Name aus deiner Sandbox (tofu output bucket_name) einsetzen:
BUCKET = "gfu-glue-training-629452195361"

## 0) Rohzustand — der Choice-Typ
Lies `customers.json` und sieh dir das abgeleitete Schema an.

In [ ]:
# customers.json direkt per S3-Pfad lesen — KEIN Katalog/Crawler nötig.
# loyalty_points ist mischtypig (1200 vs "gold") -> Glue leitet choice<> ab.
customers = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [f"s3://{BUCKET}/raw/customers/"]},
    format="json",
    transformation_ctx="customers",
)
customers.printSchema()  # loyalty_points: choice<long,string> erwarten

## 1) Auf EINEN Typ zwingen
Ersetze die `____` in der Zelle unten. Konzept: Alle Werte werden aktiv in genau
einen Zieltyp umgewandelt — nicht-numerische Werte (`"gold"`) werden dabei zu **NULL**.
- `____strategie`: welcher der vier Strings hat die Form `<verb>:<typ>` und wandelt aktiv um?

In [ ]:
# Fülle die ____ aus. Syntax steht schon — du entscheidest nur, WAS wohin gehört.
#   ____frame     : welchen DynamicFrame aus Schritt 0 löst du auf?
#   ____spalte    : welche Spalte trägt den choice<>-Konflikt (siehe Schema oben)?
#   ____strategie : welche Strategie zwingt alles auf EINEN Typ (Text -> NULL)?
#   ____ctx       : ein sprechender transformation_ctx-Name (Bookmark) für diesen Schritt
cast_long = ResolveChoice.apply(
    frame=____frame,
    specs=[("____spalte", "____strategie")],
    transformation_ctx="____ctx",
)
cast_long.printSchema()
cast_long.toDF().select("customer_id", "loyalty_points").show()

## 2) Je Typ eine eigene Spalte
Ersetze die `____`. Konzept: Aus der einen choice-Spalte werden ZWEI Spalten, eine
je Typ (`<spalte>_<typ>`) — kein Datenverlust, jeder Wert landet in seiner Spalte.
- `____strategie`: welcher String legt pro Typ eine eigene Spalte an?

In [ ]:
# Fülle die ____ aus. Syntax steht schon — du entscheidest nur, WAS wohin gehört.
#   ____frame     : welchen DynamicFrame aus Schritt 0 löst du auf?
#   ____spalte    : welche Spalte trägt den choice<>-Konflikt?
#   ____strategie : welche Strategie erzeugt je Typ eine eigene Spalte?
#   ____ctx       : ein sprechender transformation_ctx-Name (Bookmark) für diesen Schritt
make_cols = ResolveChoice.apply(
    frame=____frame,
    specs=[("____spalte", "____strategie")],
    transformation_ctx="____ctx",
)
make_cols.printSchema()
make_cols.toDF().show()

## 3) Beide Varianten in EINEM Struct
Ersetze die `____`. Konzept: Die Spalte wird zu `struct<...>`; pro Zeile ist genau
ein Ast gefüllt — ebenfalls ohne Datenverlust, nur anders verpackt als in Schritt 2.
- `____strategie`: welcher String packt beide Typen in ein gemeinsames Struct?

In [ ]:
# Fülle die ____ aus. Syntax steht schon — du entscheidest nur, WAS wohin gehört.
#   ____frame     : welchen DynamicFrame aus Schritt 0 löst du auf?
#   ____spalte    : welche Spalte trägt den choice<>-Konflikt?
#   ____strategie : welche Strategie packt beide Typen in ein Struct?
#   ____ctx       : ein sprechender transformation_ctx-Name (Bookmark) für diesen Schritt
make_struct = ResolveChoice.apply(
    frame=____frame,
    specs=[("____spalte", "____strategie")],
    transformation_ctx="____ctx",
)
make_struct.printSchema()
make_struct.toDF().show(truncate=False)

## 4) Auf EINE Variante projizieren
Ersetze die `____`. Konzept: Nur ein Ast bleibt erhalten; wo der andere Typ stand,
bleibt **NULL**. Wie „zwingen" in Schritt 1, aber OHNE Umwandlung — es wird nur ausgewählt.
- `____strategie`: welcher String hat wieder die Form `<verb>:<typ>`, wählt aber nur aus, statt umzuwandeln?

In [ ]:
# Fülle die ____ aus. Syntax steht schon — du entscheidest nur, WAS wohin gehört.
#   ____frame     : welchen DynamicFrame aus Schritt 0 löst du auf?
#   ____spalte    : welche Spalte trägt den choice<>-Konflikt?
#   ____strategie : welche Strategie behält nur einen Ast, ohne umzuwandeln?
#   ____ctx       : ein sprechender transformation_ctx-Name (Bookmark) für diesen Schritt
project_long = ResolveChoice.apply(
    frame=____frame,
    specs=[("____spalte", "____strategie")],
    transformation_ctx="____ctx",
)
project_long.printSchema()
project_long.toDF().select("customer_id", "loyalty_points").show()